# 🔑 Solutions: Lesson 7: Pipelines
**Try the problems yourself first!** Looking at a solution before struggling is like watching someone else do push-ups.
Every solution below is self-contained and has a short explanation of *why* it works.

In [1]:
import sys
sys.path.append("../utils")            # lets us import course_utils.py and agri_prep.py

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
from course_utils import *             # load_raw_farm, profile, setup_plots, ...

setup_plots()
pd.set_option("display.max_columns", 40)
pd.set_option("display.width", 140)
np.random.seed(42)


from sklearn.model_selection import GroupKFold, cross_val_score, GridSearchCV
from sklearn.pipeline import Pipeline, make_pipeline
from sklearn.compose import ColumnTransformer
from sklearn.impute import SimpleImputer
from sklearn.preprocessing import StandardScaler, OneHotEncoder, OrdinalEncoder
from sklearn.feature_selection import SelectKBest, f_regression
from sklearn.linear_model import Ridge
from agri_prep import DateFeatures, AgriFeatures, Winsorizer

df = load_lesson_clean()
LEAKY = ["total_revenue_ksh", "market_price_ksh_kg"]          # not known at planting time (Lesson 6)
X = df.drop(columns=["yield_t_ha", "farm_id", "season", "harvest_date"] + LEAKY)   # harvest_date: not known at planting
y = np.log(df["yield_t_ha"])                                  # we model log(yield)
groups = df["farm_id"]

NUM = ["rainfall_mm", "avg_temp_c", "soil_ph", "nitrogen_ppm", "phosphorus_ppm", "potassium_ppm", "organic_matter_pct",
       "fertilizer_kg_ha", "irrigation", "plant_year", "plant_month_sin", "plant_month_cos", "log_farm_size",
       "npk_total", "n_to_p_ratio", "moisture_index", "fert_x_irrigation", "rain_gap_sq", "temp_gap_sq", "ph_gap_sq"]
NOM = ["crop", "county", "soil_type", "rain_season"]
ORD = ["experience", "pest_pressure"]
ORD_CATS = [["Beginner", "Intermediate", "Advanced", "Expert"], ["Low", "Medium", "High"]]


def build_preprocessor(strategy="median", cap_k=3.0):
    return ColumnTransformer([
        ("num", Pipeline([("impute", SimpleImputer(strategy=strategy)), ("cap", Winsorizer(k=cap_k)), ("scale", StandardScaler())]), NUM),
        ("nom", Pipeline([("impute", SimpleImputer(strategy="most_frequent")),
                          ("onehot", OneHotEncoder(handle_unknown="ignore", sparse_output=False))]), NOM),
        ("ord", Pipeline([("impute", SimpleImputer(strategy="most_frequent")),
                          ("ordinal", OrdinalEncoder(categories=ORD_CATS, handle_unknown="use_encoded_value", unknown_value=-1))]), ORD),
    ], remainder="drop").set_output(transform="pandas")


def build_pipeline(model=None, k="all", strategy="median", cap_k=3.0):
    return Pipeline([("dates", DateFeatures(harvest_col=None)), ("feats", AgriFeatures()), ("prep", build_preprocessor(strategy, cap_k)),
                     ("select", SelectKBest(f_regression, k=k)), ("model", model if model is not None else Ridge(alpha=1.0))])

import joblib

---
### Problem 7.1 &nbsp; ⭐ &nbsp; Your first pipeline

Using only these raw numeric columns of `X`: `rainfall_mm, avg_temp_c, soil_ph, nitrogen_ppm, organic_matter_pct, fertilizer_kg_ha` (they contain NaN), build a pipeline **median imputer → StandardScaler → Ridge** and cross-validate it (5-fold `GroupKFold` on farms) against `y`.

Print the CV R² and the names of the pipeline steps. Why is it safe to pass raw data with NaN straight into `cross_val_score`?

In [2]:
cols = ["rainfall_mm", "avg_temp_c", "soil_ph", "nitrogen_ppm", "organic_matter_pct", "fertilizer_kg_ha"]
pipe = make_pipeline(SimpleImputer(strategy="median"), StandardScaler(), Ridge())
scores = cross_val_score(pipe, X[cols], y, cv=GroupKFold(5), groups=groups, scoring="r2")
print("steps:", list(pipe.named_steps))
print("CV R² per fold:", scores.round(3), "| mean:", scores.mean().round(3))

steps: ['simpleimputer', 'standardscaler', 'ridge']
CV R² per fold: [0.079 0.098 0.075 0.075 0.077] | mean: 0.081


**💬 Why this works:** In each fold the pipeline learns the medians and the scaling from that fold's *training* rows, then applies them to the validation rows: no leakage, no manual bookkeeping. (The score is modest because crop type, the biggest driver of yield, isn't included yet.)

---
### Problem 7.2 &nbsp; ⭐⭐ &nbsp; Mixed columns with ColumnTransformer

Build a `ColumnTransformer` that (a) median-imputes and scales `rainfall_mm`, `avg_temp_c`, `soil_ph`; (b) one-hot encodes `crop` and `county` (`handle_unknown='ignore'`); (c) encodes `pest_pressure` ordinally (Low < Medium < High) after a most-frequent imputation.

Fit on the training half of a `GroupKFold` split, transform both halves, and report the output shape, the feature names, and the number of NaN left.

In [3]:
ct = ColumnTransformer([
    ("num", make_pipeline(SimpleImputer(strategy="median"), StandardScaler()), ["rainfall_mm", "avg_temp_c", "soil_ph"]),
    ("nom", make_pipeline(SimpleImputer(strategy="most_frequent"), OneHotEncoder(handle_unknown="ignore", sparse_output=False)), ["crop", "county"]),
    ("ord", make_pipeline(SimpleImputer(strategy="most_frequent"),
                          OrdinalEncoder(categories=[["Low", "Medium", "High"]], handle_unknown="use_encoded_value", unknown_value=-1)), ["pest_pressure"]),
]).set_output(transform="pandas")

tr, te = next(GroupKFold(2).split(X, groups=groups))
Xt_train = ct.fit_transform(X.iloc[tr])          # learn on train
Xt_test  = ct.transform(X.iloc[te])              # apply to test
print("shape train/test:", Xt_train.shape, Xt_test.shape)
print("NaN left:", int(Xt_train.isna().sum().sum() + Xt_test.isna().sum().sum()))
print(list(Xt_train.columns))

shape train/test: (1164, 19) (1165, 19)
NaN left: 0
['num__rainfall_mm', 'num__avg_temp_c', 'num__soil_ph', 'nom__crop_Beans', 'nom__crop_Maize', 'nom__crop_Potato', 'nom__crop_Sorghum', 'nom__crop_Wheat', 'nom__county_Bungoma', 'nom__county_Kakamega', 'nom__county_Kiambu', 'nom__county_Kisii', 'nom__county_Machakos', 'nom__county_Meru', 'nom__county_Nakuru', 'nom__county_Nyeri', 'nom__county_Trans Nzoia', 'nom__county_Uasin Gishu', 'ord__pest_pressure']


**💬 Why this works:** Each column group gets its own recipe; the transformer remembers the training medians, means, categories and orderings, and the prefixes (`num__`, `nom__`, `ord__`) record where each output column came from.

---
### Problem 7.3 &nbsp; ⭐⭐ &nbsp; Write your own transformer

Write a scikit-learn transformer `LogColumns(columns)` that applies `np.log1p` to the given columns of a DataFrame and leaves the others untouched.

1. It must inherit `BaseEstimator` and `TransformerMixin`, store `columns` in `__init__`, implement `fit` (returning `self`) and `transform` (returning a **copy**).
2. Apply it to `farm_size_ha` and report the skewness before/after.
3. Put it into a `Pipeline` in front of a `SimpleImputer` and a `Ridge` using `['farm_size_ha','rainfall_mm']` and check that it cross-validates.

In [4]:
from sklearn.base import BaseEstimator, TransformerMixin

class LogColumns(BaseEstimator, TransformerMixin):
    """Apply log(1 + x) to the chosen columns of a DataFrame."""
    def __init__(self, columns):
        self.columns = columns
    def fit(self, X, y=None):
        return self                                   # nothing to learn
    def transform(self, X):
        X = X.copy()                                  # never modify the caller's data
        X[self.columns] = np.log1p(X[self.columns])
        return X

before = X["farm_size_ha"].skew()
after = LogColumns(["farm_size_ha"]).fit_transform(X[["farm_size_ha", "rainfall_mm"]])["farm_size_ha"].skew()
print(f"skew of farm_size_ha: {before:.2f} -> {after:.2f}")

pipe = Pipeline([("log", LogColumns(["farm_size_ha"])), ("impute", SimpleImputer(strategy="median")), ("model", Ridge())])
print("CV R²:", cross_val_score(pipe, X[["farm_size_ha", "rainfall_mm"]], y, cv=GroupKFold(5), groups=groups, scoring="r2").mean().round(3))

skew of farm_size_ha: 14.36 -> 3.26
CV R²: 0.071


**💬 Why this works:** A custom transformer is just a class with `fit` and `transform`. `__init__` only stores parameters (scikit-learn clones estimators using them), `fit` learns (here: nothing) and returns `self`, and `transform` returns a *new* object so the original data is never altered.

---
### Problem 7.4 &nbsp; ⭐⭐ &nbsp; Tune preprocessing choices with GridSearchCV

Using the `build_pipeline()` helper from the lesson setup, run a `GridSearchCV` (`GroupKFold(4)`, `scoring='r2'`) over:

- the imputation **strategy** of the numeric branch: `mean`, `median`;
- the **cap multiplier** of the Winsorizer: `1.5`, `3.0`, `10.0`;
- the Ridge **alpha**: `0.1`, `1`, `10`.

Report the best parameters and score. Which preprocessing choice mattered most (compare the best and worst combination)?

In [5]:
grid = {"prep__num__impute__strategy": ["mean", "median"],
        "prep__num__cap__k": [1.5, 3.0, 10.0],
        "model__alpha": [0.1, 1, 10]}

gs = GridSearchCV(build_pipeline(), grid, cv=GroupKFold(4), scoring="r2", n_jobs=-1).fit(X, y, groups=groups)
print("best params:", gs.best_params_)
print("best CV R² :", round(gs.best_score_, 4))

res = pd.DataFrame(gs.cv_results_).sort_values("rank_test_score")
cols = [c for c in res.columns if c.startswith("param_")] + ["mean_test_score"]
print("\nbest 3 / worst 3 combinations:")
print(pd.concat([res[cols].head(3), res[cols].tail(3)]).round(4).to_string(index=False))

best params: {'model__alpha': 1, 'prep__num__cap__k': 10.0, 'prep__num__impute__strategy': 'mean'}
best CV R² : 0.9402

best 3 / worst 3 combinations:
 param_model__alpha  param_prep__num__cap__k param_prep__num__impute__strategy  mean_test_score
                1.0                     10.0                              mean           0.9402
                0.1                     10.0                              mean           0.9401
               10.0                     10.0                              mean           0.9398
                0.1                      1.5                            median           0.9047
               10.0                      1.5                              mean           0.9047
               10.0                      1.5                            median           0.9043


**💬 Why this works:** Because imputation, capping and the model all live in one Pipeline, `GridSearchCV` can treat *preprocessing choices as hyper-parameters* and evaluate each combination without leakage. In this grid the imputation strategy and alpha hardly matter, but the Winsorizer multiplier does: capping aggressively (`k=1.5`) clips real signal from long-tailed engineered features and scores clearly worse than gentle capping (`k=10`). Preprocessing choices must be tested, not assumed.

---
### Problem 7.5 &nbsp; ⭐⭐⭐ &nbsp; Ship it: save, reload, and predict on messy new data

1. Fit `build_pipeline()` on the full training data.
2. Save it with `joblib.dump` to `../models/problem7_5.joblib`, then reload it.
3. Create **one new raw record** copied from the data, but with `county='Narok'` (never seen), `soil_ph` set to `NaN`, and `pest_pressure` set to `NaN`. Predict its yield (in t/ha) with the reloaded pipeline.
4. Confirm the reloaded pipeline gives identical predictions to the original on the first 5 rows, and explain why the prediction worked despite the problems in the new record.

In [6]:
from pathlib import Path

pipe = build_pipeline().fit(X, y)
Path("../models").mkdir(exist_ok=True)
joblib.dump(pipe, "../models/problem7_5.joblib")
loaded = joblib.load("../models/problem7_5.joblib")

new = X.iloc[[0]].copy()
new["county"] = "Narok"            # unseen category  -> one-hot gives all zeros
new["soil_ph"] = np.nan            # missing number   -> training median
new["pest_pressure"] = np.nan      # missing category -> training mode
print("prediction for the messy new record: %.2f t/ha" % np.exp(loaded.predict(new))[0])

same = np.allclose(pipe.predict(X.head(5)), loaded.predict(X.head(5)))
print("reloaded pipeline reproduces the original predictions:", same)

prediction for the messy new record: 4.67 t/ha
reloaded pipeline reproduces the original predictions: True


**💬 Why this works:** The pipeline *remembers everything learned on the training data*: medians for missing numbers, the most frequent category for missing labels, the one-hot vocabulary (with `handle_unknown='ignore'`), the Winsorizer fences, the scaler. So a raw, messy record can be sent straight in. The custom classes live in `agri_prep.py` so Python can import them when reloading: one reason real projects keep transformers in modules.